<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>



# Lab — Feature Engineering in Unity Catalog

Welcome to the "Feature Engineering in Unity Catalog" lab! In this lesson, you will learn how to load and prepare a dataset for feature selection, explore and manipulate a feature table through the Databricks UI, perform feature selection on specific columns, create a new feature table, access feature table details using both the UI and API, merge two feature tables based on a common identifier, and efficiently delete unnecessary feature tables.

**Learning Objectives:**

*By the end of this lab, you will be able to:*

- Prepare a dataset for feature selection and validate its primary key column.
- Create a feature table in Unity Catalog with `FeatureEngineeringClient`.
- Explore a feature table on the Features page.
- Retrieve feature table metadata with the API.
- Convert an existing Unity Catalog table into a feature table.
- Merge new features into an existing feature table on a common key.
- Delete a feature table that is no longer needed.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version ML V6</strong> - <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;" target="_blank">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless ML V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## Classroom Setup
Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this lab

**NOTE:** The `DA` object is only available in Databricks Academy courses.

In [0]:
%run ../Includes/Classroom-Setup-3.3

**Other Conventions:**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains **variables such as your username, catalog name, schema name, working directory, and dataset locations**. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

## Data Preparation

The classroom setup has saved the CDC data as a silver table, **`cdc_diabetes_silver`**, with a respondent key column, `UID`. The key was created **once** and stored with the data, so every feature table you build from this table shares the same keys and can be combined on `UID`.

> **Why not create the key in each notebook?** `monotonically_increasing_id()` generates IDs from how Spark happens to partition the data. If you generate it separately for two tables, or rerun the pipeline, the same row can get a different ID, and the tables no longer line up. Create a synthetic key once, persist it, and reuse it.

In [0]:
## The classroom setup saved this silver table with a stable respondent key (UID)
silver_table_name = f"{DA.catalog_name}.{DA.schema_name}.cdc_diabetes_silver"
silver_df = spark.table(silver_table_name)

display(silver_df)

## Task 1: Feature Selection

The dataset is loaded and ready. We are assuming that most of the data cleaning and feature computation is already done and data is saved to a "silver" table.

Select these features from the dataset: **"HighBP", "HighChol", "BMI", "Stroke", "PhysActivity", "GenHlth", "Sex", "Age", "Education", "Income".**

Keep the `UID` key column, drop the target column (`Diabetes_binary`), and then **validate the key**: a primary key must have no nulls and no duplicates.

In [0]:
from pyspark.sql import functions as F

## Select the key and the features we are interested in
silver_df = <FILL_IN>

## Drop the target column
silver_df = <FILL_IN>

## Validate the primary key: count null keys and duplicate keys (both must be 0)
null_keys = <FILL_IN>
duplicate_keys = <FILL_IN>
print(f"Null keys: {null_keys}, duplicate keys: {duplicate_keys}")
assert null_keys == 0 and duplicate_keys == 0, "UID must be a valid primary key"

display(silver_df)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 1: Feature Selection
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.sql import functions as F

# Select the key and the relevant feature columns
silver_df = silver_df.select(
    "UID",
    "Diabetes_binary",
    "HighBP",
    "HighChol",
    "BMI",
    "Stroke",
    "PhysActivity",
    "GenHlth",
    "Sex",
    "Age",
    "Education",
    "Income"
)

# Drop the target column
silver_df = silver_df.drop("Diabetes_binary")

# Validate the primary key: no nulls and no duplicates
null_keys = silver_df.filter(F.col("UID").isNull()).count()
duplicate_keys = silver_df.groupBy("UID").count().filter(F.col("count") > 1).count()
print(f"Null keys: {null_keys}, duplicate keys: {duplicate_keys}")
assert null_keys == 0 and duplicate_keys == 0, "UID must be a valid primary key"

display(silver_df)
```

</div>
</details>


## Task 2: Create a Feature Table

Create a feature table from the `silver_df` dataset using the `FeatureEngineeringClient`. Define a description and tags as you wish (for example, tags for the data layer, `silver`, and the source system).

The new feature table name must be **`diabetes_features`**.

**Note:** Do not define a partition column for this table.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = <FILL_IN>

diabetes_table_name = <FILL_IN>

fe.create_table(<FILL_IN>)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 2: Create Feature Table
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

diabetes_table_name = f"{DA.catalog_name}.{DA.schema_name}.diabetes_features"

fe.create_table(
    name=diabetes_table_name,
    primary_keys=["UID"],
    df=silver_df,
    description="Diabetes customer features",
    tags={"data_layer": "silver", "source_system": "cdc_diabetes", "owner": DA.username}
)
```

</div>
</details>


## Task 3: Explore Feature Table with the UI

Now that the feature table is created, visit the **Features** page from the left panel and review the following information:

* Check table columns — confirm that **`UID`** is the primary key and that no partition columns are defined.

* View **sample data**.

* View table **details**.

* View **history**.

## Task 4: Retrieve Feature Table Details

Another way to access feature table information is through the API. Use the `FeatureEngineeringClient` to retrieve and print the **`features`** and **`primary_keys`** of the table.

In [0]:
ft = fe.<FILL_IN>
print(f"Features: {ft.<FILL_IN>}")
print(f"Primary Keys: {ft.<FILL_IN>}")

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 4: Retrieve Feature Table Details
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
ft = fe.get_table(name=diabetes_table_name)
print(f"Features: {ft.features}")
print(f"Primary Keys: {ft.primary_keys}")
```

</div>
</details>

## Task 5: Create a Feature Table from an Existing UC Table

A table called **`diet_features`** has already been created for you, containing diet-related features. It was built from the same `cdc_diabetes_silver` table, so its `UID` values match the ones in `diabetes_features`. Convert this existing Delta table into a feature table by:

1. Setting the `UID` column to `NOT NULL`
2. Adding a `PRIMARY KEY` constraint on `UID`

In [0]:
display(spark.sql("SELECT * FROM diet_features"))

In [0]:
%sql
Set UID column to NOT NULL
<FILL_IN>

Add PRIMARY KEY constraint on UID
<FILL_IN>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 5: Create Feature Table from Existing UC Table
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
%sql 
ALTER TABLE diet_features ALTER COLUMN UID SET NOT NULL;
ALTER TABLE diet_features ADD CONSTRAINT diet_features_pk_constraint PRIMARY KEY(UID);
```

</div>
</details>

## Task 6: Add New Features to Existing Table

Read the `diet_features` table and add its features to the existing `diabetes_features` table. Both tables share `UID` as their primary key.

First check that the keys line up: every `UID` in `diet_features` should exist in `diabetes_features`. Then use `fe.write_table()` with `mode="merge"` and display the updated feature table.

> **Note:** `write_table(mode="merge")` is not a general join. It matches rows by the destination table's primary key, updates the feature values for matching keys, and adds rows for new keys. Rows that are not in the DataFrame you write stay unchanged.

In [0]:
diet_features = spark.sql("SELECT * FROM diet_features")

## Check that the keys line up: count diet_features keys that are missing from diabetes_features
missing_keys = <FILL_IN>
print(f"diet_features keys missing from diabetes_features: {missing_keys}")

## Merge diet features into the diabetes feature table
fe.<FILL_IN>

## Read and display the merged feature table
display(fe.<FILL_IN>)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 6: Add New Features to Existing Table
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
diet_features = spark.sql("SELECT * FROM diet_features")

# Check that the keys line up (should be 0)
missing_keys = diet_features.join(spark.table(diabetes_table_name).select("UID"), on="UID", how="left_anti").count()
print(f"diet_features keys missing from diabetes_features: {missing_keys}")

# Merge the diet features into the existing diabetes feature table
fe.write_table(
    name=diabetes_table_name,
    df=diet_features,
    mode="merge"
)

# Read and display the updated feature table
display(fe.read_table(name=diabetes_table_name))
```

</div>
</details>

## Task 7: Delete a Feature Table

After merging, the `diet_features` table is no longer needed. Use `fe.drop_table()` to delete it.

> **Sandbox cleanup only:** dropping a feature table also deletes the underlying Delta table. Only do this for the classroom-created `diet_features` table. In a real workspace, first check which models, jobs, and endpoints depend on a feature table before you delete it.

In [0]:
diet_table_name = f"{DA.catalog_name}.{DA.schema_name}.diet_features"

## Drop the feature table
fe.<FILL_IN>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 7: Delete a Feature Table
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
diet_table_name = f"{DA.catalog_name}.{DA.schema_name}.diet_features"

fe.drop_table(
    name=diet_table_name
)
```

</div>
</details>


## Conclusion

In this lab, you used Feature Engineering in Unity Catalog to perform feature engineering tasks. You loaded, prepared, and selected features from a dataset, created a feature table, explored and accessed table details through both the UI and API, merged two feature tables using a shared identifier, and deleted unnecessary tables.

This hands-on experience has enhanced your feature engineering skills on the Databricks platform.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>
